# PyTorch — Chapter 7: Your First Neural Network in PyTorch, Step by Step


## 1. Load data

- Bộ dữ liệu dùng xuyên suốt chapter: **Pima Indians Diabetes** — bài toán phân loại nhị phân (có/không mắc tiểu đường trong 5 năm), 8 biến đầu vào số học (số lần mang thai, glucose, huyết áp, độ dày da, insulin, BMI, hệ số di truyền tiểu đường, tuổi), 1 nhãn nhị phân.
- Nạp bằng `np.loadtxt(..., delimiter=',')`, tách 8 cột đầu làm `X`, cột cuối làm `y`.
- **Bắt buộc chuyển sang tensor PyTorch** trước khi dùng: `torch.tensor(X, dtype=torch.float32)`. Lý do cụ thể: PyTorch mặc định làm việc ở float32, NumPy mặc định float64.
- `y` cần `.reshape(-1, 1)`.

## 2. Định nghĩa model — 2 cách

Cùng một kiến trúc (8→12→8→1, ReLU ở 2 lớp ẩn, Sigmoid ở output) viết theo 2 cách:

- **`nn.Sequential`**: gọn, đủ dùng khi model là một chuỗi layer tuần tự thuần tuý.
- **Subclass `nn.Module`** (`class PimaClassifier`): dài hơn nhưng cần thiết khi logic forward phức tạp hơn một chuỗi tuần tự đơn giản. Bắt buộc gọi `super().__init__()` và định nghĩa `forward()`.
- Vì sao ReLU ở 2 lớp ẩn, Sigmoid ở output: Sigmoid ép output về khoảng (0,1) — dễ diễn giải thành xác suất hoặc cắt ngưỡng 0.5 để phân lớp. Sách còn nêu rõ **lý do lịch sử**: sigmoid/tanh từng dùng phổ biến ở mọi lớp, nhưng gây vanishing gradient trong mạng sâu — ReLU khắc phục vấn đề này, cho tốc độ và độ chính xác tốt hơn.

## 3. Chuẩn bị huấn luyện: loss + optimizer

- Bài toán phân loại nhị phân → `nn.BCELoss()` (binary cross-entropy).
- `optim.Adam(model.parameters(), lr=0.001)` — optimizer cần được cấp `model.parameters()` để biết tối ưu cái gì.

## 4. Huấn luyện: epoch và batch

- **Epoch**: một lượt đi qua toàn bộ tập dữ liệu. **Batch**: một nhóm mẫu được đưa vào model trong một lần cập nhật gradient.
- Cấu trúc chuẩn: 2 vòng lặp lồng nhau — vòng ngoài theo epoch, vòng trong theo batch, cắt `X`/`y` bằng slicing thủ công `X[i:i+batch_size]`.
- Sách lưu ý rõ: **kích thước batch càng lớn** thì mỗi bước cập nhật tốn tính toán hơn, đổi lại ít bước hơn mỗi epoch.

## 5. Đánh giá model

- `with torch.no_grad(): y_pred = model(X)` — tắt theo dõi gradient khi chỉ cần suy luận, tiết kiệm bộ nhớ/tính toán.
- Accuracy = tỉ lệ `y_pred.round() == y` đúng — làm tròn xác suất sigmoid thành nhãn 0/1 rồi so khớp.
- Neural network là thuật toán **stochastic**: cùng code, cùng data, mỗi lần chạy ra kết quả khác nhau. Sách minh hoạ bằng 5 lần chạy: accuracy dao động quanh 77% (`0.760`–`0.784`).

## 6. Dự đoán

- `predictions = model(X)` rồi `.round()` — hoặc `(model(X) > 0.5).int()` để ra thẳng nhãn 0/1.


## 7. Vận dụng


**7.3** — Nạp CSV và chuyển NumPy array sang tensor PyTorch

In [1]:
import numpy as np
import torch

# load the dataset, split into input (X) and output (y) variables
dataset = np.loadtxt('pima-indians-diabetes.csv', delimiter=',')
X = dataset[:,0:8]
y = dataset[:,8]

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)


**7.10** — Code hoàn chỉnh: định nghĩa model (Sequential) + huấn luyện + đánh giá

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# load the dataset, split into input (X) and output (y) variables
dataset = np.loadtxt('pima-indians-diabetes.csv', delimiter=',')
X = dataset[:,0:8]
y = dataset[:,8]

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

# define the model
model = nn.Sequential(
    nn.Linear(8, 12),
    nn.ReLU(),
    nn.Linear(12, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)
print(model)

# train the model
loss_fn   = nn.BCELoss()  # binary cross entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)

n_epochs = 100
batch_size = 10

for epoch in range(n_epochs):
    for i in range(0, len(X), batch_size):
        Xbatch = X[i:i+batch_size]
        y_pred = model(Xbatch)
        ybatch = y[i:i+batch_size]
        loss = loss_fn(y_pred, ybatch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Finished epoch {epoch}, latest loss {loss}')

# compute accuracy (no_grad is optional)
with torch.no_grad():
    y_pred = model(X)
accuracy = (y_pred.round() == y).float().mean()
print(f"Accuracy {accuracy}")


Sequential(
  (0): Linear(in_features=8, out_features=12, bias=True)
  (1): ReLU()
  (2): Linear(in_features=12, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=1, bias=True)
  (5): Sigmoid()
)
Finished epoch 0, latest loss 0.531125009059906
Finished epoch 1, latest loss 0.49733084440231323
Finished epoch 2, latest loss 0.49044615030288696
Finished epoch 3, latest loss 0.4808984100818634
Finished epoch 4, latest loss 0.4631302058696747
Finished epoch 5, latest loss 0.47402533888816833
Finished epoch 6, latest loss 0.471670538187027
Finished epoch 7, latest loss 0.46505963802337646
Finished epoch 8, latest loss 0.46508848667144775
Finished epoch 9, latest loss 0.46817001700401306
Finished epoch 10, latest loss 0.46515530347824097
Finished epoch 11, latest loss 0.4594789147377014
Finished epoch 12, latest loss 0.4587650001049042
Finished epoch 13, latest loss 0.45658859610557556
Finished epoch 14, latest loss 0.46128714084625244
Finished epoch 15, lates

**7.13** — Code hoàn chỉnh: định nghĩa model (subclass nn.Module) + huấn luyện + dự đoán 5 mẫu

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# load the dataset, split into input (X) and output (y) variables
dataset = np.loadtxt('pima-indians-diabetes.csv', delimiter=',')
X = dataset[:,0:8]
y = dataset[:,8]

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

# define the model
class PimaClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(8, 12)
        self.act1 = nn.ReLU()
        self.hidden2 = nn.Linear(12, 8)
        self.act2 = nn.ReLU()
        self.output = nn.Linear(8, 1)
        self.act_output = nn.Sigmoid()

    def forward(self, x):
        x = self.act1(self.hidden1(x))
        x = self.act2(self.hidden2(x))
        x = self.act_output(self.output(x))
        return x

model = PimaClassifier()
print(model)

# train the model
loss_fn   = nn.BCELoss()  # binary cross entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)

n_epochs = 100
batch_size = 10

for epoch in range(n_epochs):
    for i in range(0, len(X), batch_size):
        Xbatch = X[i:i+batch_size]
        y_pred = model(Xbatch)
        ybatch = y[i:i+batch_size]
        loss = loss_fn(y_pred, ybatch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# compute accuracy
y_pred = model(X)
accuracy = (y_pred.round() == y).float().mean()
print(f"Accuracy {accuracy}")

# make class predictions with the model
predictions = (model(X) > 0.5).int()
for i in range(5):
    print('%s => %d (expected %d)' % (X[i].tolist(), predictions[i], y[i]))


PimaClassifier(
  (hidden1): Linear(in_features=8, out_features=12, bias=True)
  (act1): ReLU()
  (hidden2): Linear(in_features=12, out_features=8, bias=True)
  (act2): ReLU()
  (output): Linear(in_features=8, out_features=1, bias=True)
  (act_output): Sigmoid()
)
Accuracy 0.7747395634651184
[6.0, 148.0, 72.0, 35.0, 0.0, 33.599998474121094, 0.6269999742507935, 50.0] => 1 (expected 1)
[1.0, 85.0, 66.0, 29.0, 0.0, 26.600000381469727, 0.35100001096725464, 31.0] => 0 (expected 0)
[8.0, 183.0, 64.0, 0.0, 0.0, 23.299999237060547, 0.671999990940094, 32.0] => 1 (expected 1)
[1.0, 89.0, 66.0, 23.0, 94.0, 28.100000381469727, 0.16699999570846558, 21.0] => 0 (expected 0)
[0.0, 137.0, 40.0, 35.0, 168.0, 43.099998474121094, 2.2880001068115234, 33.0] => 1 (expected 1)
